[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/doxav/astromodel_proving/blob/main/analysis/01_postfit_sqlite_pipeline.ipynb)


In [ ]:
# Colab / local repository setup
# Run this cell first when opening the notebook in Google Colab. It clones the
# repository, installs the pinned requirements, and makes data/ plus src/ imports
# available from the same execution context used by the local notebooks.
from pathlib import Path
import os
import subprocess
import sys


def _running_in_colab() -> bool:
    return "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules


def _run(command, cwd=None):
    print("$", " ".join(command))
    subprocess.run(command, cwd=cwd, check=True)


REPO_URL = os.environ.get("ASTROMODEL_REPO_URL", "https://github.com/doxav/astromodel_proving.git")
REPO_BRANCH = os.environ.get("ASTROMODEL_REPO_BRANCH", "main")
PROJECT_DIRNAME = os.environ.get("ASTROMODEL_PROJECT_DIRNAME", "astromodel_proving")

if _running_in_colab():
    project_root = Path("/content") / PROJECT_DIRNAME
    if not project_root.exists():
        clone_cmd = [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            REPO_BRANCH,
            REPO_URL,
            str(project_root),
        ]
        try:
            _run(clone_cmd)
        except subprocess.CalledProcessError:
            # Some forks/default branches may not be named like REPO_BRANCH.
            # Retry without an explicit branch before surfacing the clone error.
            _run(["git", "clone", "--depth", "1", REPO_URL, str(project_root)])
    os.chdir(project_root)
    requirements = project_root / "requirements.txt"
    if requirements.exists():
        _run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)])
else:
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]
    project_root = next(
        (
            candidate
            for candidate in candidates
            if (candidate / "src").is_dir() and (candidate / "data").is_dir()
        ),
        current,
    )
    os.chdir(project_root)

os.environ["ASTROMODEL_PROJECT_ROOT"] = str(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"ASTROMODEL_PROJECT_ROOT={project_root}")
print(f"Working directory={Path.cwd()}")
print(f"data exists={(project_root / 'data').exists()}, src exists={(project_root / 'src').exists()}")


# Step 01 — SQLite post-fit pipeline and hidden-mechanism summaries

This notebook reruns the **legacy single-current post-fit summaries** used for reviewer-response context.

## Reuse from `astro_atf_analysis_improved_sectioned.ipynb`

None of the ATF parsing, preprocessing, or sweep-feature extraction logic is reused here.  
This step interprets the **historical Optuna DB summaries**, not the new ATF dataset.

## Interpretation boundary

Because historical control provenance remains unresolved in step 00, the outputs below should be treated as:
- structural-confounding diagnostics,
- effective-parameter summaries,
- and provisional mechanism narratives,

not as final reviewer-facing mechanism proof.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path(os.environ.get("ASTROMODEL_PROJECT_ROOT", Path.cwd())).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.postfit_sqlite import d_pk_invariance_check, run_step01_postfit_sqlite

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)

PROJECT_ROOT

In [ ]:
results = run_step01_postfit_sqlite(PROJECT_ROOT)
top_trials_df = results["top_trials_all_dbs"]
effective_df = results["effective_parameter_summary"]
representative_df = results["representative_mechanism_summary"]

print("written outputs:", sorted((PROJECT_ROOT / "outputs" / "postfit_sqlite").glob("*.csv")))
print("top-trial rows:", len(top_trials_df), "effective rows:", len(effective_df), "representative rows:", len(representative_df))

## `d × pk` confounding check

In [ ]:
base_params = {
    "gki": 50.0,
    "pk": 2e-4,
    "d": 3.0,
    "gt": 8.0,
    "gs": 10.0,
    "zth": 0.2,
    "zs": 0.05,
    "K_bath_value_middle": 8.2,
    "eps": 0.01,
    "eps_middle": 0.5,
    "wo": 1500.0,
    "ca": 400.0,
    "gl_a": 0.01,
    "Va_l": -70.0,
    "Va_s": -90.0,
    "switching_function": "sigmoid",
}
check = d_pk_invariance_check(base_params)
display(
    pd.DataFrame(
        [
            {
                "P_gap_eff_a": check.P_gap_eff_a,
                "P_gap_eff_b": check.P_gap_eff_b,
                "I_kgap_a": check.I_kgap_a,
                "I_kgap_b": check.I_kgap_b,
                "max_abs_rhs_delta": float(abs(check.dzdt_a - check.dzdt_b).max()),
            }
        ]
    )
)

## Effective parameter summary (all 18 DBs)

In [ ]:
display(effective_df)

fig, ax = plt.subplots(figsize=(10, 4))
for condition, group in effective_df.groupby("condition"):
    ax.plot(group["current_na"], group["P_gap_eff"], marker="o", label=condition)
ax.set_xlabel("Current (nA)")
ax.set_ylabel("P_gap_eff")
ax.set_title("Effective gap permeability across historical best trials")
ax.legend()
plt.show()

## Representative mechanism summary

In [ ]:
display(representative_df)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(representative_df["condition"], representative_df["gap_to_kir_integral_ratio"])
ax.set_ylabel("gap_to_kir_integral_ratio")
ax.set_title("Representative mechanism contrast by condition")
plt.show()

## Consequence for the reviewer response

Step 01 still supports the mechanistic framing that **effective combinations** matter more than raw `d` or `pk` alone.  
However, because it is built on legacy single-current fits — and because historical control provenance is still unresolved — the strongest reviewer-facing evidence among steps 00-02 remains the ATF-based step 02 thresholds and region summaries.

## Post-execution scientific status

Executed status for reviewer response: Step 01 remains legacy single-current context. It produced 18 effective-parameter summary rows and 3 representative hidden-mechanism summaries, supporting the R1/R4 argument that raw parameters should be interpreted through effective combinations such as `P_gap_eff`. Because these fits are historical single-current SQLite results and inherit provenance limitations from Step 00, they are not used as reviewer-facing proof of biological degeneracy or phenotype classification.